In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Data Reading

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@storageaccnitin.dfs.core.windows.net/products")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")
df.display()

### **Functions**

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
create or replace function dbcatalog.bronze.discount_function(pricce double)
returns double
language sql
return pricce * 0.90 

In [0]:
%sql
select product_id, dbcatalog.bronze.discount_function(price) as dis_price, price from products

In [0]:
df = df.withColumn("discounted_price", expr("dbcatalog.bronze.discount_function(price)"))
df.display()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@storageaccnitin.dfs.core.windows.net/products")

In [0]:
spark.read.format("delta").load("abfss://silver@storageaccnitin.dfs.core.windows.net/products").display()

In [0]:
%sql
create table if not exists dbcatalog.silver.products_silver
using delta
location 'abfss://silver@storageaccnitin.dfs.core.windows.net/products'

In [0]:
%sql
select * from dbcatalog.silver.products_silver